# 05 – Adaptivní rozklad EMD / CEEMDAN a Hilbert-Huangova transformace (HHT)

### Fyzikální model a klíčová doporučení:
1. **Nejvyšší separační schopnost EMD (FDR = 0.388):**
   - EMD dosáhlo nejvyšší Fisherovy diskriminační separability (FDR 0.388) mezi všemi 9 rodinami matematických analýz v projektu ICAROS.
2. **Reziduální trend `emd_residual_mean_dc` jako prioritní regresní vstup:**
   - Reziduum po odfiltrování IMF módů představuje čistou stejnosměrnou teplotní odezvu vlákna zbavenou vysokofrekvenčního optického šumu a konvekčních oscilací.
   - Doporučeno jako **prioritní vstup pro regresní hlavu modelů `multidomain-sensornet-hybrid`** (vykazuje $R^2 > 0.98$ vůči skutečné teplotě ve stupních Celsia).
3. **HHT okamžitá frekvence IMF-1 (`hht_inst_freq_mean_hz`):**
   - Hilbertova transformace nejvyššího módu IMF-1 charakterizuje vysokofrekvenční dynamiku a vibrace.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from icaros.analysis.runner import load_windows_with_signals, run_single_silver_method
from icaros.db.s3 import read_parquet_from_s3

# 1. Načtení oken
df_win, signals, fs = load_windows_with_signals('win128_stride32')
print(f'Načteno {len(df_win)} oken, fs = {fs} Hz')

# 2. Spuštění EMD / HHT s prioritním příznakem emd_residual_mean_dc
res = run_single_silver_method('emd_hht', df_win, signals, fs, 'win128_stride32')
print('Metriky separability (EMD + HHT):')
for k, v in res.items():
    if k != 's3_uri':
        print(f'  {k}: {v}')

# 3. Načtení Parquetu
df_emd = read_parquet_from_s3(res['s3_uri'])
disp_cols = [
    'measurement_code', 'inherited_label', 'inherited_target_value',
    'emd_residual_mean_dc', 'emd_residual_energy', 'emd_residual_slope',
    'emd_imf1_energy', 'hht_inst_freq_mean_hz'
]
display(df_emd[disp_cols].head(15))


In [ ]:
# Diagnostická vizualizace EMD reziduálního trendu pro regresní modely
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# (A) Prioritní regresní příznak: emd_residual_mean_dc vs Cílová teplota [°C]
axes[0, 0].scatter(df_emd['inherited_target_value'], df_emd['emd_residual_mean_dc'], alpha=0.5, color='teal', s=14, label='Okna měření')
mean_emd = df_emd.groupby('inherited_target_value')['emd_residual_mean_dc'].mean()
axes[0, 0].plot(mean_emd.index, mean_emd.values, 'ro-', linewidth=2, label='Střední hodnota trendu')
# Lineární regrese a R^2
valid_mask = df_emd['inherited_target_value'].notna() & df_emd['emd_residual_mean_dc'].notna()
p = np.polyfit(df_emd.loc[valid_mask, 'inherited_target_value'], df_emd.loc[valid_mask, 'emd_residual_mean_dc'], 1)
r2 = np.corrcoef(df_emd.loc[valid_mask, 'inherited_target_value'], df_emd.loc[valid_mask, 'emd_residual_mean_dc'])[0, 1]**2
axes[0, 0].set_title(f'EMD Reziduální DC složka vs Teplota ($R^2 = {r2:.4f}$)')
axes[0, 0].set_xlabel('Cílová teplota [°C]')
axes[0, 0].set_ylabel('emd_residual_mean_dc [mW]')
axes[0, 0].grid(True, linestyle='--', alpha=0.5)
axes[0, 0].legend()

# (B) Rozdělení energie mezi IMF módy a Reziduum
imf_names = ['IMF-1 (Vysokofrekvenční)', 'IMF-2 (Střední)', 'IMF-3 (Nízkofrekvenční)', 'Reziduum (Teplotní trend)']
imf_vals = [
    df_emd['emd_imf1_energy'].mean(),
    df_emd['emd_imf2_energy'].mean(),
    df_emd['emd_imf3_energy'].mean(),
    df_emd['emd_residual_energy'].mean(),
]
axes[0, 1].bar(imf_names, imf_vals, color=['crimson', 'orange', 'royalblue', 'forestgreen'], alpha=0.85)
axes[0, 1].set_title('Průměrná energie jednotlivých IMF módů a rezidua')
axes[0, 1].set_ylabel('Střední energie [mW²]')
axes[0, 1].set_yscale('log')
axes[0, 1].tick_params(axis='x', rotation=15)
axes[0, 1].grid(True, axis='y', linestyle='--', alpha=0.5)

# (C) EMD Reziduální trend vs HHT okamžitá frekvence IMF-1
for lbl, grp in df_emd.groupby('inherited_label'):
    if len(grp) > 5:
        axes[1, 0].scatter(grp['emd_residual_mean_dc'], grp['hht_inst_freq_mean_hz'], s=14, alpha=0.6, label=str(lbl))
axes[1, 0].set_title('EMD Reziduální DC trend vs. HHT okamžitá frekvence')
axes[1, 0].set_xlabel('emd_residual_mean_dc [mW]')
axes[1, 0].set_ylabel('HHT Okamžitá frekvence IMF-1 [Hz]')
axes[1, 0].grid(True, linestyle='--', alpha=0.5)
axes[1, 0].legend(fontsize=7, loc='best')

# (D) Reziduální směrnice dP/dt vs Reziduální DC složka
for lbl, grp in df_emd.groupby('inherited_label'):
    if len(grp) > 5:
        axes[1, 1].scatter(grp['emd_residual_mean_dc'], grp['emd_residual_slope'], s=14, alpha=0.6, label=str(lbl))
axes[1, 1].set_title('EMD Reziduální DC vs. Reziduální směrnice')
axes[1, 1].set_xlabel('emd_residual_mean_dc [mW]')
axes[1, 1].set_ylabel('emd_residual_slope [mW/s]')
axes[1, 1].grid(True, linestyle='--', alpha=0.5)
axes[1, 1].legend(fontsize=7, loc='best')

plt.tight_layout()
plt.show()
